## Code for Finetuning

In [ ]:
#!/usr/bin/env python

import os, json, torch, warnings
from PIL import Image
from torch.utils.data import Dataset
from transformers import (
    LlavaProcessor,
    LlavaForConditionalGeneration,
    TrainingArguments,
    Trainer,
)
from tqdm import TqdmWarning

warnings.filterwarnings("ignore", category=TqdmWarning)
os.environ["TORCHINDUCTOR_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"


# ── Config ────────────────────────────────────────────────────────────────
TRAIN_JSON = "./RadSpineXR/Train/llava_dataset/train/dataset.json"
VAL_JSON   = "./RadSpineXR/Train/llava_dataset/validation/dataset.json"
IMAGE_DIR  = "./RadSpineXR/Train/llava_dataset/images"
OUTPUT_DIR = "./llava_phi_finetuned_spine"

MODEL_ID   = "xtuner/llava-phi-3-mini-hf"
EPOCHS     = 5
BSZ        = 4
GRAD_ACC   = 4
LR         = 5e-5
EVAL_STEPS = 100

# ── Load Model and Processor ──────────────────────────────────────────────
processor = LlavaProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
processor.patch_size = 14  # ✅ Explicit fix for vision tower

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",  # ✅ Enables multi-GPU
    trust_remote_code=True
)

try:
    model.tie_weights()
except Exception as e:
    print(f"[tie_weights warning] {e}", flush=True)

# ── Freeze vision tower ───────────────────────────────────────────────────
for name, param in model.named_parameters():
    if name.startswith("vision_tower"):
        param.requires_grad = False

# ── Dataset ───────────────────────────────────────────────────────────────
class SpineXRLLaVADataset(Dataset):
    def __init__(self, json_path, image_folder, processor):
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        self.samples = []
        for item in data:
            img_path = os.path.join(image_folder, item["image"])
            convs = item["conversations"]
            for user, assistant in zip(convs[0::2], convs[1::2]):
                self.samples.append({
                    "image_path": img_path,
                    "question": user["value"],
                    "answer": assistant["value"]
                })
        self.processor = processor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s["image_path"]).convert("RGB")
        prompt = "<|user|>\n<image>\n" + s["question"] + "\n<|end|>\n<|assistant|>\n"

        enc_all = self.processor(
            text=prompt + s["answer"],
            images=img,
            return_tensors="pt",
            padding="max_length",
        )
        input_ids = enc_all.input_ids.squeeze(0)
        attention_mask = enc_all.attention_mask.squeeze(0)
        pixel_values = enc_all.pixel_values.squeeze(0)

        enc_prompt = self.processor(
            text=prompt,
            images=img,
            return_tensors="pt",
            padding="max_length",
        )
        prompt_len = (enc_prompt.input_ids != self.processor.tokenizer.pad_token_id).sum().item()

        labels = input_ids.clone()
        labels[:prompt_len] = -100
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

# ── Load Datasets ──────────────────────────────────────────────────────────
train_ds = SpineXRLLaVADataset(TRAIN_JSON, IMAGE_DIR, processor)
val_ds   = SpineXRLLaVADataset(VAL_JSON, IMAGE_DIR, processor)

# ── Collate ────────────────────────────────────────────────────────────────
def collate_fn(batch):
    return {
        "pixel_values":   torch.stack([x["pixel_values"]   for x in batch]),
        "input_ids":      torch.stack([x["input_ids"]      for x in batch]),
        "attention_mask": torch.stack([x["attention_mask"] for x in batch]),
        "labels":         torch.stack([x["labels"]         for x in batch]),
    }

# ── Custom Trainer ─────────────────────────────────────────────────────────
class MyTrainer(Trainer):
    def training_step(self, model, inputs, *args, **kwargs):
        loss = super().training_step(model, inputs, *args, **kwargs)
        if not torch.isfinite(loss):
            print(f"⚠️ NaN detected at step {self.state.global_step}", flush=True)
            return torch.tensor(0.0, requires_grad=True).to(loss.device)
        print(f"[step {self.state.global_step:>4}] train_loss={loss.item():.4f}", flush=True)
        return loss

# ── Training Arguments ─────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BSZ,
    per_device_eval_batch_size=BSZ,
    gradient_accumulation_steps=GRAD_ACC,
    num_train_epochs=EPOCHS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_steps=100,
    logging_steps=10,
    learning_rate=LR,
    warmup_steps=100,
    weight_decay=0.01,
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    fp16=False,
    bf16=True,
    save_total_limit=2,
    remove_unused_columns=False,
    logging_dir="./logs",
    report_to="none",
    dataloader_num_workers=4,
)

# ── Train ──────────────────────────────────────────────────────────────────
trainer = MyTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
)

print("🚀 Starting fine-tuning...", flush=True)
trainer.train()

# ── Save ───────────────────────────────────────────────────────────────────
print("✅ Training complete. Saving model...", flush=True)
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("🎉 Done!", flush=True)


## Code For Zeroshot


In [ ]:
import os
import json
from pathlib import Path

import torch
from PIL import Image
import pandas as pd
from tqdm import tqdm

from transformers import LlavaProcessor, LlavaForConditionalGeneration

# ───────────────────────── CONFIG (EDIT THESE) ─────────────────────────
# JSON format assumed same as your train/val:
# [
#   {
#     "image": "vindr_train_013.png",
#     "conversations": [
#       {"from": "human", "value": "Q1 ..."},
#       {"from": "gpt",   "value": "A1 ..."},
#       {"from": "human", "value": "Q2 ..."},
#       {"from": "gpt",   "value": "A2 ..."},
#       ...
#     ]
#   },
#   ...
# ]

TEST_JSON = "./RadSpineXR/Test/sample_150.json"          # <-- put your test json here
IMAGE_DIR = "./RadSpineXR/Test/Unannoated_images_150" # or a separate test image dir
OUT_CSV   = "./llava_phi_zeroshot_test.csv"

# Base model OR your finetuned checkpoint:
#   - base zero-shot:    "xtuner/llava-phi-3-mini-hf"
#   - finetuned testing: "./llava_phi_finetuned_spine"
MODEL_ID  = "xtuner/llava-phi-3-mini-hf"

MAX_NEW_TOKENS = 256
TEMPERATURE    = 0.0
TOP_P          = 1.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ───────────────────────── Load model & processor ────────────────────────
processor = LlavaProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

# Match your training fix
processor.patch_size = 14

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map=None,
    trust_remote_code=True,
).to(DEVICE)
model.eval()

print("Model and processor loaded from:", MODEL_ID)

# ───────────────────────── Load test samples ─────────────────────────────
def load_spinexr_samples(json_path: str | Path, image_dir: str | Path):
    json_path = Path(json_path)
    image_dir = Path(image_dir)

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    samples = []
    for item in data:
        img_path = image_dir / item["image"]
        convs = item.get("conversations", [])

        # Same flattening logic as in your training dataset:
        # (human, gpt), (human, gpt), ...
        for user, assistant in zip(convs[0::2], convs[1::2]):
            q = user.get("value", "").strip()
            a = assistant.get("value", "").strip() if assistant is not None else ""

            samples.append(
                {
                    "image_path": str(img_path),
                    "question": q,
                    "answer": a,  # ground truth (if present)
                }
            )

    return samples

samples = load_spinexr_samples(TEST_JSON, IMAGE_DIR)
print(f"Loaded {len(samples)} QA pairs from test JSON")

# ───────────────────────── Generation helper ─────────────────────────────
def generate_answer(image_path: str, question: str) -> str:
    img = Image.open(image_path).convert("RGB")

    # Same prompt template as training
    prompt = "<|user|>\n<image>\n" + question.strip() + "\n<|end|>\n<|assistant|>\n"

    inputs = processor(
        text=prompt,
        images=img,
        return_tensors="pt",
    )

    # Move to device
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    # Some models don't have pad_token_id set; fall back to eos if needed
    pad_id = processor.tokenizer.pad_token_id
    if pad_id is None:
        pad_id = processor.tokenizer.eos_token_id

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=False,
            pad_token_id=pad_id,
        )[0]

    # Strip the prompt tokens to keep only generated continuation
    input_len = inputs["input_ids"].shape[1]
    gen_ids = output_ids[input_len:]

    text = processor.tokenizer.decode(gen_ids, skip_special_tokens=True)
    return text.strip()

# ───────────────────────── Run zero-shot over test set ───────────────────
rows = []
for idx, s in enumerate(tqdm(samples, desc="Running zero-shot")):
    img_path = s["image_path"]
    question = s["question"]
    gt_answer = s["answer"]  # may be "" if not available

    try:
        pred = generate_answer(img_path, question)
    except Exception as e:
        print(f"[ERROR] idx={idx}, image={img_path}, question={question[:50]}... -> {e}")
        pred = "[ERROR] " + str(e)

    rows.append(
        {
            "image_path": img_path,
            "question": question,
            "ground_truth": gt_answer,
            "prediction": pred,
        }
    )

# ───────────────────────── Save to CSV ────────────────────────────────────
df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False, encoding="utf-8")
print(f"✅ Saved predictions to: {OUT_CSV}")
